In [2]:
import sys
sys.path.insert(0, "../")

### diagnose_missing_crops

In [ ]:
# See why input images did not make a crop.
from pathlib import Path
from ultralytics import YOLO
from config.config import config
from src.detection import crop_detections, _passes_aspect_ratio_filter

model = YOLO(str(config.yolo100ep_best_weights))
source_dir = config.simulation_data_dir / "images"

image_paths = list(source_dir.glob("*.jpg")) + list(source_dir.glob("*.png"))
no_crop_images = []

for img_path in image_paths:
    crops, _ = crop_detections(model, img_path)
    if not crops:
        no_crop_images.append(img_path.name)

print(f"\n{len(no_crop_images)} images produced zero usable crops:\n")

# Now re-run raw YOLO (no filtering at all) on just those, to see WHY
for name in no_crop_images:
    img_path = source_dir / name
    results = model(str(img_path), verbose=False)
    boxes = [(float(b.conf[0]), b.xyxy[0].tolist()) for r in results for b in r.boxes]
    if not boxes:
        print(f"{name}: NO DETECTION AT ALL — YOLO found nothing above its internal threshold")
    else:
        best_conf, best_box = max(boxes, key=lambda x: x[0])
        w = best_box[2] - best_box[0]
        h = best_box[3] - best_box[1]
        ratio = w / h if h else 0
        below_conf_threshold = best_conf < config.yolo_confidence_threshold
        too_small = h < config.yolo_min_crop_height_px
        print(f"{name}: best_conf={best_conf:.2f} (threshold={config.yolo_confidence_threshold}), "
              f"aspect_ratio={ratio:.2f}, height_px={h:.0f} (min={config.yolo_min_crop_height_px}), "
              f"below_conf={below_conf_threshold}, too_small={too_small}")


16 images produced zero usable crops:

001.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
009.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
010.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
013.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
019.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
025.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
037.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
048.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
050.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
054.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
057.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
058.jpg: NO DETECTION AT ALL — YOLO found nothing above its internal threshold
060.jpg: NO 

### Build Whitelist

In [ ]:
"""
Step 2 of setting up the simulation whitelist.

Reads data/simulation/whitelist_labels.csv (produced by
scripts/prepare_simulation_crops.py — every reviewed crop's TRUE plate text,
hand-corrected via the labeling UI) and lets you interactively pick which ones
become the whitelist.

Why this is a separate step from labeling itself: labeling covers every crop
that was produced (~68 in your case), but the whitelist should only be a
subset (~20) — the rest exist in the simulation pool as "unknown" vehicles the
system should correctly flag as not-whitelisted. Keeping selection separate
from labeling means you can change which plates are whitelisted later without
re-doing any manual transcription work.

Output: config/whitelist.json — {plate_text: True} used by the whitelist
matcher (fuzzy matching logic, not written yet — this just produces the data
it will read).

"""
import csv
import json
from pathlib import Path

from config.config import config

LABELS_CSV = config.simulation_data_dir / "whitelist_labels.csv"
WHITELIST_JSON = Path().resolve().parent / "config" / "whitelist.json"


def load_labels():
    if not LABELS_CSV.exists():
        raise FileNotFoundError(
            f"{LABELS_CSV} not found — run scripts/prepare_simulation_crops.py first."
        )
    with open(LABELS_CSV, newline="") as f:
        return list(csv.DictReader(f))


def main():
    rows = load_labels()
    # corrected_text is the TRUE plate (what you typed), not predicted_text —
    # see prepare_simulation_crops.py docstring for why this matters: whitelisting
    # the OCR's own prediction would make every match trivially "correct" and
    # tell you nothing about whether fuzzy matching actually works.
    usable = [r for r in rows if r.get("corrected_text", "").strip()]

    print(f"{len(usable)} labeled plates available.\n")
    for i, r in enumerate(usable):
        print(f"[{i}] {r['filename']}: {r['corrected_text']}  "
              f"(predicted: {r['predicted_text']}, was_correct: {r['was_correct']})")

    print("\nEnter the numbers of plates to whitelist, comma-separated (e.g. 0,3,5,12).")
    print("Aim for ~20. Mix of clean/easy reads and a few that were originally")
    print("mis-predicted (was_correct=False) so fuzzy matching actually gets exercised.")
    selection = input("> ").strip()

    indices = [int(x.strip()) for x in selection.split(",") if x.strip()]
    chosen = {usable[i]["corrected_text"]: True for i in indices}

    WHITELIST_JSON.parent.mkdir(parents=True, exist_ok=True)
    with open(WHITELIST_JSON, "w") as f:
        json.dump(chosen, f, indent=2)

    print(f"\n{len(chosen)} plates written to {WHITELIST_JSON}:")
    for plate in chosen:
        print(f"  {plate}")

main()

63 labeled plates available.

[0] 002_plate0_conf0.91.jpg: MP09CP9052  (predicted: MP09CP9052, was_correct: True)
[1] 003_plate0_conf0.50.jpg: KA18P2987  (predicted: , was_correct: False)
[2] 004_plate0_conf0.94.jpg: MH12JC2813  (predicted: MH12JC2813, was_correct: True)
[3] 005_plate0_conf0.93.jpg: KL43B2344  (predicted: KL43B2344, was_correct: True)
[4] 006_plate0_conf0.92.jpg: UP50AS4535  (predicted: UP50AS4535, was_correct: True)
[5] 007_plate0_conf0.49.jpg: RJ27TC0530  (predicted: RJ27TC0530, was_correct: True)
[6] 008_plate0_conf0.93.jpg: MH46X9996  (predicted: MH46X9996, was_correct: True)
[7] 011_plate0_conf0.93.jpg: MH20BY3665  (predicted: MH20BY3665, was_correct: True)
[8] 012_plate0_conf0.93.jpg: MH20CS1941  (predicted: MH20CS19, was_correct: False)
[9] 014_plate0_conf0.93.jpg: HR26BP3543  (predicted: HR26BP3543, was_correct: True)
[10] 015_plate0_conf0.94.jpg: DL10CG4693  (predicted: DL10CG4693, was_correct: True)
[11] 016_plate0_conf0.90.jpg: MH06AW8929  (predicted: MH06AW